# 401 · Protobuf wire format playground

Companion to [Protobuf wire format](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/401/protobuf-wire-format/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/401/wire_format_playground.ipynb)

**Goal:** encode tags, varints, length-delimited fields, nested messages, and unpacked repeated fields by hand—then check against golden hex.

**Why this lab:** API tutorials hide the wire. Debugging hex dumps, evolution, and subset codecs requires operational wire literacy.

**How to use:** run steps **in order** (later cells reuse helpers). Each step asserts against known hex.

**Expect:** every assert prints `OK …` with matching bytes; G1–G5 match the course golden table.

> **Honesty banner:** numbers and timings in these notebooks are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth for this project. Notebooks teach mechanisms, not leaderboards.


## Setup

Stdlib only. No `protoc` required for this playground.

**Why:** wire rules are independent of any one language runtime.

**How:** define hex helpers used by later asserts.

**Expect:** no output beyond definitions until the first encode step.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional, Tuple


def hex_bytes(b: bytes) -> str:
    return " ".join(f"{x:02x}" for x in b)


def assert_hex(actual: bytes, expected_hex: str, label: str = "") -> None:
    exp = bytes.fromhex(expected_hex.replace(" ", ""))
    assert actual == exp, f"{label}: got {hex_bytes(actual)!r}, expected {hex_bytes(exp)!r}"
    print(f"OK {label}: {hex_bytes(actual)}")



## 1. Encode a key (tag)

**Why:** every field on the wire starts with a key: field number + wire type. Names never appear.

**How:** `key = (field_number << 3) | wire_type`, then encode as a varint.

**Expect:** field 1 VARINT → `08`; field 2 LEN → `12`; field 16 VARINT → `80 01` (multi-byte key).

| Wire type | Value |
|-----------|------:|
| VARINT | 0 |
| I64 | 1 |
| LEN | 2 |
| I32 | 5 |

**Why it matters:** reading a hex dump starts with splitting keys from payloads.


In [ ]:
def encode_varint(u: int) -> bytes:
    if u < 0:
        raise ValueError("unsigned varint only in this lab subset")
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(field_number: int, wire_type: int) -> bytes:
    return encode_varint((field_number << 3) | wire_type)


# Field 1 VARINT → 0x08; field 2 LEN → 0x12; field 16 VARINT → 0x80 0x01
assert_hex(encode_key(1, 0), "08", "field1 varint key")
assert_hex(encode_key(2, 2), "12", "field2 len key")
assert_hex(encode_key(16, 0), "80 01", "field16 varint key")



## 2. Encode varints (base-128)

**Why:** integers use a compact variable-length encoding with a continuation bit.

**How:** while value ≥ 128 emit `(value & 0x7f) | 0x80`, shift; last byte high bit clear.

**Expect:** `1→01`, `127→7f`, `128→80 01`, `300→ac 02`.

**Why it matters:** small ids stay one byte; large values grow—and truncated varints are a decoder hazard.


In [ ]:
for n, hx in [(1, "01"), (127, "7f"), (128, "80 01"), (300, "ac 02")]:
    assert_hex(encode_varint(n), hx, f"varint {n}")



## 3. Decode varint with bounds

**Why:** hostile or truncated input must not read off the end of the buffer.

**How:** decode with a max of 10 continuation bytes; reject EOF mid-varint.

**Expect:** `ac 02` → 300; lone `80` raises `WireError`.

**Why it matters:** bounds checks are part of being a real decoder (see also 301 untrusted input).


In [ ]:
class WireError(Exception):
    pass


def decode_varint(buf: bytes, i: int = 0) -> Tuple[int, int]:
    value = 0
    shift = 0
    bytes_read = 0
    while True:
        if i >= len(buf):
            raise WireError("truncated varint")
        b = buf[i]
        i += 1
        bytes_read += 1
        if bytes_read > 10:
            raise WireError("overlong varint")
        value |= (b & 0x7F) << shift
        if (b & 0x80) == 0:
            break
        shift += 7
    return value, i


assert decode_varint(bytes.fromhex("ac 02"))[0] == 300
try:
    decode_varint(bytes.fromhex("80"))  # continues, EOF
    raise AssertionError("expected WireError")
except WireError as e:
    print("OK truncated varint:", e)



## 4. Length-delimited string field

**Why:** strings, bytes, and nested messages share wire type LEN (2).

**How:** stage UTF-8 payload, emit key, length varint, then bytes—length never after payload on the wire.

**Expect:** `name="Ada"` field 2 → `12 03 41 64 61`.

**Why it matters:** wrong length is how truncated messages and buffer overruns show up.


In [ ]:
def encode_string_field(field_number: int, s: str) -> bytes:
    payload = s.encode("utf-8")
    return encode_key(field_number, 2) + encode_varint(len(payload)) + payload


# name="Ada" on field 2 → 12 03 41 64 61
assert_hex(encode_string_field(2, "Ada"), "12 03 41 64 61", "string Ada")



## 5. Nested message + unpacked repeated

**Why:** nesting is not a special wire type—only LEN whose payload is another message. Repeated unpacked = one full field per element.

**How:** build `manager={id=2}` and `tags=[1,2]` with the helpers above.

**Expect:** nested → `1a 02 08 02`; tags → `20 01 20 02`.

**Why it matters:** you can now read real nested Protobuf hex without guessing.


In [ ]:
def encode_uint32_field(field_number: int, v: int) -> bytes:
    if v == 0:
        return b""  # proto3 omit default
    return encode_key(field_number, 0) + encode_varint(v)


# manager={id=2} on field 3 → 1a 02 08 02
inner = encode_uint32_field(1, 2)
nested = encode_key(3, 2) + encode_varint(len(inner)) + inner
assert_hex(nested, "1a 02 08 02", "nested manager id=2")

# tags=[1,2] field 4 unpacked → 20 01 20 02
tags = b"".join(encode_key(4, 0) + encode_varint(t) for t in (1, 2))
assert_hex(tags, "20 01 20 02", "unpacked tags")



## 6. MiniUser G1–G5 goldens (preview of the lab)

**Why:** the full 401 lab uses fixed golden vectors so implementations are comparable across languages.

**How:** encode MiniUser cases G1–G5 (proto3 omit defaults).

**Expect:** hex matches the table in the lab article (`G1` Ada, empty `G2`, `300`, tags, nested manager).

Teaching schema (not the suite `benchmark_v2.proto`):

```protobuf
message MiniUser {
  uint32 id = 1;
  string name = 2;
  MiniUser manager = 3;
  repeated uint32 tags = 4;
}
```

**Why it matters:** golden bytes beat “looks right in a debugger.”


In [ ]:
@dataclass
class MiniUser:
    id: int = 0
    name: str = ""
    manager: Optional["MiniUser"] = None
    tags: List[int] = field(default_factory=list)


def encode_mini_user(u: MiniUser) -> bytes:
    out = bytearray()
    out += encode_uint32_field(1, u.id)
    if u.name:
        out += encode_string_field(2, u.name)
    if u.manager is not None:
        inner = encode_mini_user(u.manager)
        out += encode_key(3, 2) + encode_varint(len(inner)) + inner
    for t in u.tags:
        out += encode_key(4, 0) + encode_varint(t)
    return bytes(out)


GOLDENS = {
    "G1": (MiniUser(id=1, name="Ada"), "08 01 12 03 41 64 61"),
    "G2": (MiniUser(), ""),
    "G3": (MiniUser(id=300), "08 ac 02"),
    "G4": (MiniUser(tags=[1, 2]), "20 01 20 02"),
    "G5": (MiniUser(manager=MiniUser(id=2)), "1a 02 08 02"),
}

for label, (user, hx) in GOLDENS.items():
    raw = encode_mini_user(user)
    if hx == "":
        assert raw == b"", label
        print(f"OK {label}: <empty>")
    else:
        assert_hex(raw, hx, label)



## Next

- Full lab (decode, unknown skip, bounds, official oracle): [lab_mini_protobuf_encoder.ipynb](./lab_mini_protobuf_encoder.ipynb)
- Article: [Protobuf wire format](../../401/protobuf-wire-format.md)

**Why continue:** encoding alone is not enough—decoders must skip unknowns and fail closed on truncation.
